In [155]:
import pandas as pd
import os 
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt

In [156]:
os.chdir("S:/23_010_CVD_trial_emu")

In [157]:
#import THIN extract which relates to people who have at least one prescription for empagliflozin or DPP-4i between 1/1/14 - 31/12/22
df = pd.read_csv('person.txt', sep='\t')

In [133]:
#import other datafiles
thin = pd.read_csv("thin_codelist_dictionary.txt", sep=',', low_memory=False)
#med_list = pd.read_csv("thin_product_dictionary.txt", sep=',', low_memory=False)
bio_change = pd.read_csv('RD_person_biologic_changes.txt', sep=',', low_memory=False)
smoking = pd.read_csv('RD_person_attributes.txt', sep=',')
eth_count = pd.read_csv('eth_count.csv')
measure = pd.read_csv('RD_person_measure_changes.txt', sep=',', low_memory=False)

In [158]:
#import outcome data 
death = pd.read_csv("date_of_death.csv")
df = pd.merge(df, death, how='left', on='person_id')

In [159]:
def died_binary_function(row): 
    
    if row['year_of_death']==0: 
        return(0)
    else: 
        return(1)
    
df['died'] = df.apply(died_binary_function, axis=1)

In [160]:
#rename death date 
df = df.rename(columns={"death_date":"corrected_death_date"})

In [161]:
#correct the outcome column so that the correct last contact date is the correct date of death, if death occurred 
def last_contact_correct_function(row): 
    if row['died']==1: 
        return(row['corrected_death_date'])
    else: 
        return(row['last_contact_date'])
    
df['last_contact_correct'] = df.apply(last_contact_correct_function, axis=1)

In [162]:
#import care site data 
cs = pd.read_csv("S:/23_010_CVD_trial_emu/march_analysis/all_contact_care_siteid.txt")
cs = cs.drop_duplicates(subset=['person_id', 'care_site_id', 'start_date'])

In [ ]:
#test if people have multiple different care sites 
res_dict = {}

for person in list(cs['person_id'].unique()):
    number_care_sites = len(cs[cs['person_id']==person]['care_site_id'].unique()) 
    print(person, number_care_sites)
    res_dict[person]=number_care_sites

In [ ]:
#no patients had a recorded change of GP practice 
pd.DataFrame.from_dict(res_dict, orient='index')[0].value_counts(dropna=False)

In [163]:
#merge duplicates in cs
cs = cs.groupby("person_id").agg("first").reset_index()

In [164]:
df = df.sort_values("person_id")
cs = cs.sort_values("person_id")

In [165]:
#merge care site id 
df = pd.merge(df, cs, how='left', on='person_id')

In [166]:
#import imd data 
imd_deciles = pd.read_csv("S:/23_010_CVD_trial_emu/THIN_IMD_2023_10_30.csv")

In [167]:
#merge imd data at GP practice level for each individual 
df = pd.merge(df, imd_deciles, how='left', on='care_site_id')

In [168]:
#import prescription data 
complete_pres = pd.read_csv('complete_pres.csv', low_memory=False, usecols=['start_date', 'person_id', 'short_name', 'product_id', 'dosage_1'])

In [169]:
#import diagnostic codes 
first_obs = pd.read_csv('obs_min_dates.csv', low_memory=False)
first_pmh = pd.read_csv('pmh_min_dates.csv', low_memory=False)
first_contact = pd.read_csv('contact_min_dates.csv', low_memory=False)

In [170]:
#import drug codelists
any_metformin_codes = list(pd.read_csv('drug_names/any_metformin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
metformin_monotherapy_codes = list(pd.read_csv('drug_names/metformin_monotherapy_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
contraindicated_metformin_codes = list(pd.read_csv('drug_names/contraindicated_metformin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

sglt2i_codes = list(pd.read_csv('drug_names/sglt2i_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
dpp4i_codes = list(pd.read_csv('drug_names/dpp4i_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

empa_codes = list(pd.read_csv('drug_names/empa_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
dapa_codes = list(pd.read_csv('drug_names/dapagliflozin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
ertu_codes = list(pd.read_csv('drug_names/ertugliflozin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
cana_codes = list(pd.read_csv('drug_names/canagliflozin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

insulin_codes = list(pd.read_csv('drug_names/insulin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
su_codes = list(pd.read_csv('drug_names/su_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
glp1ra_codes = list(pd.read_csv('drug_names/glp1ra_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
tzd_codes = list(pd.read_csv('drug_names/tzd_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

sita_codes = list(pd.read_csv('drug_names/sitagliptin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
saxa_codes = list(pd.read_csv('drug_names/saxagliptin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
lina_codes = list(pd.read_csv('drug_names/linagliptin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
vilda_codes = list(pd.read_csv('drug_names/vildagliptin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
alo_codes = list(pd.read_csv('drug_names/alogliptin_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

anti_htn_codes = list(pd.read_csv('drug_names/anti_htn_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])
lipid_lowering_codes = list(pd.read_csv('drug_names/anti_lipids_product_id.csv').drop(columns=['Unnamed: 0'])['product_id'])

clopi_codes = list(pd.read_csv('S:/23_010_CVD_trial_emu/drug_names/clopi_product_id.csv')['product_id'])
dipyrid_codes = list(pd.read_csv('S:/23_010_CVD_trial_emu/drug_names/dipyridamole_product_id.csv')['product_id'])
anti_platelet_codes = clopi_codes + dipyrid_codes

anticoag_codes = list(pd.read_csv('S:/23_010_CVD_trial_emu/drug_names/anticoag_product_id.csv')['product_id'])

In [171]:
#calculate the earliest date of prescription for a drug 
any_metformin_first_dates = complete_pres[complete_pres['product_id'].isin(any_metformin_codes)].groupby('person_id')['start_date'].min()
metformin_monotherapy_first_dates = complete_pres[complete_pres['product_id'].isin(metformin_monotherapy_codes)].groupby('person_id')['start_date'].min()
contraindicated_metformin_first_dates = complete_pres[complete_pres['product_id'].isin(contraindicated_metformin_codes)].groupby('person_id')['start_date'].min()

sglt2i_first_dates = complete_pres[complete_pres['product_id'].isin(sglt2i_codes)].groupby('person_id')['start_date'].min()
dpp4i_first_dates = complete_pres[complete_pres['product_id'].isin(dpp4i_codes)].groupby('person_id')['start_date'].min()

empa_first_dates = complete_pres[complete_pres['product_id'].isin(empa_codes)].groupby('person_id')['start_date'].min()
dapa_first_dates = complete_pres[complete_pres['product_id'].isin(dapa_codes)].groupby('person_id')['start_date'].min()
ertu_first_dates = complete_pres[complete_pres['product_id'].isin(ertu_codes)].groupby('person_id')['start_date'].min()
cana_first_dates = complete_pres[complete_pres['product_id'].isin(cana_codes)].groupby('person_id')['start_date'].min()

insulin_first_dates = complete_pres[complete_pres['product_id'].isin(insulin_codes)].groupby('person_id')['start_date'].min()
su_first_dates = complete_pres[complete_pres['product_id'].isin(su_codes)].groupby('person_id')['start_date'].min()
tzd_first_dates = complete_pres[complete_pres['product_id'].isin(tzd_codes)].groupby('person_id')['start_date'].min()
glp1ra_first_dates = complete_pres[complete_pres['product_id'].isin(glp1ra_codes)].groupby('person_id')['start_date'].min()

sita_first_dates = complete_pres[complete_pres['product_id'].isin(sita_codes)].groupby('person_id')['start_date'].min()
saxa_first_dates = complete_pres[complete_pres['product_id'].isin(saxa_codes)].groupby('person_id')['start_date'].min()
lina_first_dates = complete_pres[complete_pres['product_id'].isin(lina_codes)].groupby('person_id')['start_date'].min()
vilda_first_dates = complete_pres[complete_pres['product_id'].isin(vilda_codes)].groupby('person_id')['start_date'].min()
alo_first_dates = complete_pres[complete_pres['product_id'].isin(alo_codes)].groupby('person_id')['start_date'].min()

anti_htn_first_dates = complete_pres[complete_pres['product_id'].isin(anti_htn_codes)].groupby('person_id')['start_date'].min()
lipid_lowering_first_dates = complete_pres[complete_pres['product_id'].isin(lipid_lowering_codes)].groupby('person_id')['start_date'].min()

anti_platelet_first_dates = complete_pres[complete_pres['product_id'].isin(anti_platelet_codes)].groupby('person_id')['start_date'].min()
anticoag_first_dates = complete_pres[complete_pres['product_id'].isin(anticoag_codes)].groupby('person_id')['start_date'].min()

In [172]:
#calculate the latest date for a prescription for a drug 
any_metformin_last_dates = complete_pres[complete_pres['product_id'].isin(any_metformin_codes)].groupby('person_id')['start_date'].max()
metformin_monotherapy_last_dates = complete_pres[complete_pres['product_id'].isin(metformin_monotherapy_codes)].groupby('person_id')['start_date'].max()
contraindicated_metformin_last_dates = complete_pres[complete_pres['product_id'].isin(contraindicated_metformin_codes)].groupby('person_id')['start_date'].max()

sglt2i_last_dates = complete_pres[complete_pres['product_id'].isin(sglt2i_codes)].groupby('person_id')['start_date'].max()
dpp4i_last_dates = complete_pres[complete_pres['product_id'].isin(dpp4i_codes)].groupby('person_id')['start_date'].max()

empa_last_dates = complete_pres[complete_pres['product_id'].isin(empa_codes)].groupby('person_id')['start_date'].max()
dapa_last_dates = complete_pres[complete_pres['product_id'].isin(dapa_codes)].groupby('person_id')['start_date'].max()
ertu_last_dates = complete_pres[complete_pres['product_id'].isin(ertu_codes)].groupby('person_id')['start_date'].max()
cana_last_dates = complete_pres[complete_pres['product_id'].isin(cana_codes)].groupby('person_id')['start_date'].max()

insulin_last_dates = complete_pres[complete_pres['product_id'].isin(insulin_codes)].groupby('person_id')['start_date'].max()
su_last_dates = complete_pres[complete_pres['product_id'].isin(su_codes)].groupby('person_id')['start_date'].max()
tzd_last_dates = complete_pres[complete_pres['product_id'].isin(tzd_codes)].groupby('person_id')['start_date'].max()
glp1ra_last_dates = complete_pres[complete_pres['product_id'].isin(glp1ra_codes)].groupby('person_id')['start_date'].max()

sita_last_dates = complete_pres[complete_pres['product_id'].isin(sita_codes)].groupby('person_id')['start_date'].max()
saxa_last_dates = complete_pres[complete_pres['product_id'].isin(saxa_codes)].groupby('person_id')['start_date'].max()
lina_last_dates = complete_pres[complete_pres['product_id'].isin(lina_codes)].groupby('person_id')['start_date'].max()
vilda_last_dates = complete_pres[complete_pres['product_id'].isin(vilda_codes)].groupby('person_id')['start_date'].max()
alo_last_dates = complete_pres[complete_pres['product_id'].isin(alo_codes)].groupby('person_id')['start_date'].max()

anti_htn_last_dates = complete_pres[complete_pres['product_id'].isin(anti_htn_codes)].groupby('person_id')['start_date'].max()
lipid_lowering_last_dates = complete_pres[complete_pres['product_id'].isin(lipid_lowering_codes)].groupby('person_id')['start_date'].max()

anti_platelet_last_dates = complete_pres[complete_pres['product_id'].isin(anti_platelet_codes)].groupby('person_id')['start_date'].max()
anticoag_last_dates = complete_pres[complete_pres['product_id'].isin(anticoag_codes)].groupby('person_id')['start_date'].max()

In [173]:
#merge date of first prescription to the main dataframe 

#any_metformin
any_metformin_first_dates_df = pd.DataFrame(any_metformin_first_dates).reset_index()
any_metformin_first_dates_df.columns = ['person_id', 'any_metformin_first_dates']
df = pd.merge(df, any_metformin_first_dates_df, how='left', on='person_id')

#metformin_monotherapy
metformin_monotherapy_first_dates_df = pd.DataFrame(metformin_monotherapy_first_dates).reset_index()
metformin_monotherapy_first_dates_df.columns = ['person_id', 'metformin_monotherapy_first_dates']
df = pd.merge(df, metformin_monotherapy_first_dates_df, how='left', on='person_id')

#contraindicated_metformin
contraindicated_metformin_first_dates_df = pd.DataFrame(contraindicated_metformin_first_dates).reset_index()
contraindicated_metformin_first_dates_df.columns = ['person_id', 'contraindicated_metformin_first_dates']
df = pd.merge(df, contraindicated_metformin_first_dates_df, how='left', on='person_id')

#sglt2i
sglt2i_first_dates_df = pd.DataFrame(sglt2i_first_dates).reset_index()
sglt2i_first_dates_df.columns = ['person_id', 'sglt2i_first_dates']
df = pd.merge(df, sglt2i_first_dates_df, how='left', on='person_id')

#dpp4i
dpp4i_first_dates_df = pd.DataFrame(dpp4i_first_dates).reset_index()
dpp4i_first_dates_df.columns = ['person_id', 'dpp4i_first_dates']
df = pd.merge(df, dpp4i_first_dates_df, how='left', on='person_id')

#empa
empa_first_dates_df = pd.DataFrame(empa_first_dates).reset_index()
empa_first_dates_df.columns = ['person_id', 'empa_first_dates']
df = pd.merge(df, empa_first_dates_df, how='left', on='person_id')

#dapa
dapa_first_dates_df = pd.DataFrame(dapa_first_dates).reset_index()
dapa_first_dates_df.columns = ['person_id', 'dapa_first_dates']
df = pd.merge(df, dapa_first_dates_df, how='left', on='person_id')

#ertu
ertu_first_dates_df = pd.DataFrame(ertu_first_dates).reset_index()
ertu_first_dates_df.columns = ['person_id', 'ertu_first_dates']
df = pd.merge(df, ertu_first_dates_df, how='left', on='person_id')

#cana
cana_first_dates_df = pd.DataFrame(cana_first_dates).reset_index()
cana_first_dates_df.columns = ['person_id', 'cana_first_dates']
df = pd.merge(df, cana_first_dates_df, how='left', on='person_id')

#insulin
insulin_first_dates_df = pd.DataFrame(insulin_first_dates).reset_index()
insulin_first_dates_df.columns = ['person_id', 'insulin_first_dates']
df = pd.merge(df, insulin_first_dates_df, how='left', on='person_id')

#sita
sita_first_dates_df = pd.DataFrame(sita_first_dates).reset_index()
sita_first_dates_df.columns = ['person_id', 'sita_first_dates']
df = pd.merge(df, sita_first_dates_df, how='left', on='person_id')

#saxa
saxa_first_dates_df = pd.DataFrame(saxa_first_dates).reset_index()
saxa_first_dates_df.columns = ['person_id', 'saxa_first_dates']
df = pd.merge(df, saxa_first_dates_df, how='left', on='person_id')

#lina
lina_first_dates_df = pd.DataFrame(lina_first_dates).reset_index()
lina_first_dates_df.columns = ['person_id', 'lina_first_dates']
df = pd.merge(df, lina_first_dates_df, how='left', on='person_id')

#vilda
vilda_first_dates_df = pd.DataFrame(vilda_first_dates).reset_index()
vilda_first_dates_df.columns = ['person_id', 'vilda_first_dates']
df = pd.merge(df, vilda_first_dates_df, how='left', on='person_id')

#alo
alo_first_dates_df = pd.DataFrame(alo_first_dates).reset_index()
alo_first_dates_df.columns = ['person_id', 'alo_first_dates']
df = pd.merge(df, alo_first_dates_df, how='left', on='person_id')

#anti_htn
anti_htn_first_dates_df = pd.DataFrame(anti_htn_first_dates).reset_index()
anti_htn_first_dates_df.columns = ['person_id', 'anti_htn_first_dates']
df = pd.merge(df, anti_htn_first_dates_df, how='left', on='person_id')

#lipid_lowering
lipid_lowering_first_dates_df = pd.DataFrame(lipid_lowering_first_dates).reset_index()
lipid_lowering_first_dates_df.columns = ['person_id', 'lipid_lowering_first_dates']
df = pd.merge(df, lipid_lowering_first_dates_df, how='left', on='person_id')

#su
su_first_dates_df = pd.DataFrame(su_first_dates).reset_index()
su_first_dates_df.columns = ['person_id', 'su_first_dates']
df = pd.merge(df, su_first_dates_df, how='left', on='person_id')

#glp1ra
glp1ra_first_dates_df = pd.DataFrame(glp1ra_first_dates).reset_index()
glp1ra_first_dates_df.columns = ['person_id', 'glp1ra_first_dates']
df = pd.merge(df, glp1ra_first_dates_df, how='left', on='person_id')

#tzd
tzd_first_dates_df = pd.DataFrame(tzd_first_dates).reset_index()
tzd_first_dates_df.columns = ['person_id', 'tzd_first_dates']
df = pd.merge(df, tzd_first_dates_df, how='left', on='person_id')

#antiplatelet
anti_platelet_dates_df = pd.DataFrame(anti_platelet_first_dates).reset_index()
anti_platelet_dates_df.columns = ['person_id', 'anti_platelet_first_dates']
df = pd.merge(df, anti_platelet_dates_df, how='left', on='person_id')

#anticoag
anticoag_dates_df = pd.DataFrame(anticoag_first_dates).reset_index()
anticoag_dates_df.columns = ['person_id', 'anticoag_first_dates']
df = pd.merge(df, anticoag_dates_df, how='left', on='person_id')

In [174]:
#merge date of last prescription to the main dataframe 

#any_metformin
any_metformin_last_dates_df = pd.DataFrame(any_metformin_last_dates).reset_index()
any_metformin_last_dates_df.columns = ['person_id', 'any_metformin_last_dates']
df = pd.merge(df, any_metformin_last_dates_df, how='left', on='person_id')

#metformin_monotherapy
metformin_monotherapy_last_dates_df = pd.DataFrame(metformin_monotherapy_last_dates).reset_index()
metformin_monotherapy_last_dates_df.columns = ['person_id', 'metformin_monotherapy_last_dates']
df = pd.merge(df, metformin_monotherapy_last_dates_df, how='left', on='person_id')

#contraindicated_metformin
contraindicated_metformin_last_dates_df = pd.DataFrame(contraindicated_metformin_last_dates).reset_index()
contraindicated_metformin_last_dates_df.columns = ['person_id', 'contraindicated_metformin_last_dates']
df = pd.merge(df, contraindicated_metformin_last_dates_df, how='left', on='person_id')

#sglt2i
sglt2i_last_dates_df = pd.DataFrame(sglt2i_last_dates).reset_index()
sglt2i_last_dates_df.columns = ['person_id', 'sglt2i_last_dates']
df = pd.merge(df, sglt2i_last_dates_df, how='left', on='person_id')

#dpp4i
dpp4i_last_dates_df = pd.DataFrame(dpp4i_last_dates).reset_index()
dpp4i_last_dates_df.columns = ['person_id', 'dpp4i_last_dates']
df = pd.merge(df, dpp4i_last_dates_df, how='left', on='person_id')

#empa
empa_last_dates_df = pd.DataFrame(empa_last_dates).reset_index()
empa_last_dates_df.columns = ['person_id', 'empa_last_dates']
df = pd.merge(df, empa_last_dates_df, how='left', on='person_id')

#dapa
dapa_last_dates_df = pd.DataFrame(dapa_last_dates).reset_index()
dapa_last_dates_df.columns = ['person_id', 'dapa_last_dates']
df = pd.merge(df, dapa_last_dates_df, how='left', on='person_id')

#ertu
ertu_last_dates_df = pd.DataFrame(ertu_last_dates).reset_index()
ertu_last_dates_df.columns = ['person_id', 'ertu_last_dates']
df = pd.merge(df, ertu_last_dates_df, how='left', on='person_id')

#cana
cana_last_dates_df = pd.DataFrame(cana_last_dates).reset_index()
cana_last_dates_df.columns = ['person_id', 'cana_last_dates']
df = pd.merge(df, cana_last_dates_df, how='left', on='person_id')

#insulin
insulin_last_dates_df = pd.DataFrame(insulin_last_dates).reset_index()
insulin_last_dates_df.columns = ['person_id', 'insulin_last_dates']
df = pd.merge(df, insulin_last_dates_df, how='left', on='person_id')

#sita
sita_last_dates_df = pd.DataFrame(sita_last_dates).reset_index()
sita_last_dates_df.columns = ['person_id', 'sita_last_dates']
df = pd.merge(df, sita_last_dates_df, how='left', on='person_id')

#saxa
saxa_last_dates_df = pd.DataFrame(saxa_last_dates).reset_index()
saxa_last_dates_df.columns = ['person_id', 'saxa_last_dates']
df = pd.merge(df, saxa_last_dates_df, how='left', on='person_id')

#lina
lina_last_dates_df = pd.DataFrame(lina_last_dates).reset_index()
lina_last_dates_df.columns = ['person_id', 'lina_last_dates']
df = pd.merge(df, lina_last_dates_df, how='left', on='person_id')

#vilda
vilda_last_dates_df = pd.DataFrame(vilda_last_dates).reset_index()
vilda_last_dates_df.columns = ['person_id', 'vilda_last_dates']
df = pd.merge(df, vilda_last_dates_df, how='left', on='person_id')

#alo
alo_last_dates_df = pd.DataFrame(alo_last_dates).reset_index()
alo_last_dates_df.columns = ['person_id', 'alo_last_dates']
df = pd.merge(df, alo_last_dates_df, how='left', on='person_id')

#anti_htn
anti_htn_last_dates_df = pd.DataFrame(anti_htn_last_dates).reset_index()
anti_htn_last_dates_df.columns = ['person_id', 'anti_htn_last_dates']
df = pd.merge(df, anti_htn_last_dates_df, how='left', on='person_id')

#lipid_lowering
lipid_lowering_last_dates_df = pd.DataFrame(lipid_lowering_last_dates).reset_index()
lipid_lowering_last_dates_df.columns = ['person_id', 'lipid_lowering_last_dates']
df = pd.merge(df, lipid_lowering_last_dates_df, how='left', on='person_id')

#su
su_last_dates_df = pd.DataFrame(su_last_dates).reset_index()
su_last_dates_df.columns = ['person_id', 'su_last_dates']
df = pd.merge(df, su_last_dates_df, how='left', on='person_id')

#antiplatelet
anti_platelet_dates_df = pd.DataFrame(anti_platelet_last_dates).reset_index()
anti_platelet_dates_df.columns = ['person_id', 'anti_platelet_last_dates']
df = pd.merge(df, anti_platelet_dates_df, how='left', on='person_id')

#anticoag
anticoag_dates_df = pd.DataFrame(anticoag_last_dates).reset_index()
anticoag_dates_df.columns = ['person_id', 'anticoag_last_dates']
df = pd.merge(df, anticoag_dates_df, how='left', on='person_id')

In [175]:
#set datetime format for the first prescription date 
df['any_metformin_first_dates'] = pd.to_datetime(df['any_metformin_first_dates'])
df['metformin_monotherapy_first_dates'] = pd.to_datetime(df['metformin_monotherapy_first_dates'])
df['contraindicated_metformin_first_dates'] = pd.to_datetime(df['contraindicated_metformin_first_dates'])
df['sglt2i_first_dates'] = pd.to_datetime(df['sglt2i_first_dates'])
df['empa_first_dates'] = pd.to_datetime(df['empa_first_dates'])
df['dpp4i_first_dates'] = pd.to_datetime(df['dpp4i_first_dates'])
df['cana_first_dates'] = pd.to_datetime(df['cana_first_dates'])
df['dapa_first_dates'] = pd.to_datetime(df['dapa_first_dates'])
df['ertu_first_dates'] = pd.to_datetime(df['ertu_first_dates'])
df['sita_first_dates'] = pd.to_datetime(df['sita_first_dates'])
df['lina_first_dates'] = pd.to_datetime(df['lina_first_dates'])
df['alo_first_dates'] = pd.to_datetime(df['alo_first_dates'])
df['saxa_first_dates'] = pd.to_datetime(df['saxa_first_dates'])
df['vilda_first_dates'] = pd.to_datetime(df['vilda_first_dates'])
df['insulin_first_dates'] = pd.to_datetime(df['insulin_first_dates'])
df['anti_htn_first_dates'] = pd.to_datetime(df['anti_htn_first_dates'])
df['lipid_lowering_first_dates'] = pd.to_datetime(df['lipid_lowering_first_dates'])
df['su_first_dates'] = pd.to_datetime(df['su_first_dates'])
df['glp1ra_first_dates'] = pd.to_datetime(df['glp1ra_first_dates'])
df['anti_platelet_first_dates'] = pd.to_datetime(df['anti_platelet_first_dates'])
df['anticoag_first_dates'] = pd.to_datetime(df['anticoag_first_dates'])

In [176]:
#set datetime format for the last prescription date 
df['any_metformin_last_dates'] = pd.to_datetime(df['any_metformin_last_dates'])
df['metformin_monotherapy_last_dates'] = pd.to_datetime(df['metformin_monotherapy_last_dates'])
df['contraindicated_metformin_last_dates'] = pd.to_datetime(df['contraindicated_metformin_last_dates'])
df['sglt2i_last_dates'] = pd.to_datetime(df['sglt2i_last_dates'])
df['empa_last_dates'] = pd.to_datetime(df['empa_last_dates'])
df['dpp4i_last_dates'] = pd.to_datetime(df['dpp4i_last_dates'])
df['cana_last_dates'] = pd.to_datetime(df['cana_last_dates'])
df['dapa_last_dates'] = pd.to_datetime(df['dapa_last_dates'])
df['ertu_last_dates'] = pd.to_datetime(df['ertu_last_dates'])
df['sita_last_dates'] = pd.to_datetime(df['sita_last_dates'])
df['lina_last_dates'] = pd.to_datetime(df['lina_last_dates'])
df['alo_last_dates'] = pd.to_datetime(df['alo_last_dates'])
df['saxa_last_dates'] = pd.to_datetime(df['saxa_last_dates'])
df['vilda_last_dates'] = pd.to_datetime(df['vilda_last_dates'])
df['insulin_last_dates'] = pd.to_datetime(df['insulin_last_dates'])
df['anti_htn_last_dates'] = pd.to_datetime(df['anti_htn_last_dates'])
df['lipid_lowering_last_dates'] = pd.to_datetime(df['lipid_lowering_last_dates'])
df['su_last_dates'] = pd.to_datetime(df['su_last_dates'])
df['anti_platelet_last_dates'] = pd.to_datetime(df['anti_platelet_last_dates'])
df['anticoag_last_dates'] = pd.to_datetime(df['anticoag_last_dates'])

In [177]:
#exclude prevalent users of either drug 
def define_study_cohort(row):
    """ 
    Assigns cohort based on the first study drug prescription 
    while excluding prevalent users of either SGLT2i or DPP-4i.
    This ensures only SGLT2i and DPP-4i naive people are included in the cohort. 
    It also excludes people if they have an iambigious exposure status 
    e.g commencing two SGLT2i on same day as either first DPP-4i/empa, commencing both empa and DPP4i on same day. 
    """

    # Get first recorded prescription of either empagliflozin or DPP-4i
    first_study_drug_date = min(
        [row['empa_first_dates'], row['dpp4i_first_dates']], 
        key=lambda x: x if pd.notna(x) else datetime(9999, 1, 1)
    )

    # If both are missing, exclude
    if pd.isna(first_study_drug_date) or first_study_drug_date == datetime(9999, 1, 1):
        return 'excluded - both missing'

    # Exclude prevalent users: Anyone who took an SGLT2i or DPP-4i before their first study drug
    if (not pd.isna(row['sglt2i_first_dates']) and row['sglt2i_first_dates'] < first_study_drug_date) or \
       (not pd.isna(row['dpp4i_first_dates']) and row['dpp4i_first_dates'] < first_study_drug_date) or \
       (not pd.isna(row['sglt2i_first_dates']) and row['sglt2i_first_dates'] < pd.to_datetime('2014-01-01')) or \
       (not pd.isna(row['dpp4i_first_dates']) and row['dpp4i_first_dates'] < pd.to_datetime('2014-01-01')):
        return 'excluded - prior SGLT2i or DPP-4i use'

    # Exclude if SGLT2i and DPP-4i were initiated on the same day
    if not pd.isna(row['empa_first_dates']) and not pd.isna(row['dpp4i_first_dates']) and \
       row['sglt2i_first_dates'] == row['dpp4i_first_dates'] == row['empa_first_dates']:
        return 'excluded - both SGLT2i and DPP4-i started same day'
    
    # Exclude if multiple SGLT2i initiated on cohort entry day
    if (not pd.isna(row['cana_first_dates']) and row['cana_first_dates'] == first_study_drug_date): 
        return 'excluded - multiple SGLT2i commenced on the same day'
    elif (not pd.isna(row['dapa_first_dates']) and row['dapa_first_dates'] == first_study_drug_date): 
        return 'excluded - multiple SGLT2i commenced on the same day'
    elif (not pd.isna(row['ertu_first_dates']) and row['ertu_first_dates'] == first_study_drug_date): 
        return 'excluded - multiple SGLT2i commenced on the same day'   

    # Assign cohort based on which drug was initiated first
    if first_study_drug_date == row['empa_first_dates']:
        return 'included - empa'
    elif first_study_drug_date == row['dpp4i_first_dates']:
        return 'included - dpp4i'

    return 'excluded - unknown case'  # Catch any undefined cases

# Apply function to dataset
df['cohort'] = df.apply(define_study_cohort, axis=1)



In [ ]:
df['cohort'].value_counts(dropna=False)

In [179]:
df = df[(df['cohort']=='included - dpp4i')|(df['cohort']=='included - empa')]

In [ ]:
df['cohort'].value_counts(dropna=False)

In [181]:
#define cohort entry date
def cohort_entry_date_function(row): 
    print('working on', row.name)
    
    if row['cohort'] == 'included - empa': 
        return(row['empa_first_dates'])
    else: 
        return(row['dpp4i_first_dates'])

In [ ]:
df['cohort_entry_date'] = df.apply(cohort_entry_date_function, axis=1)

In [ ]:
#prescribed contraindicated metformin combination pill before or on cohort entry date 
def combined_therapy_exclusion_function(row): 
    print('working on', row.name)
    if row['contraindicated_metformin_first_dates']<=row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)

df['combined_therapy_exclusion'] = df.apply(combined_therapy_exclusion_function, axis=1)

In [ ]:
df['combined_therapy_exclusion'].value_counts(dropna=False)

In [185]:
df = df[df['combined_therapy_exclusion']==0]

In [ ]:
df.shape

In [187]:
#people who last contact date occurred before 1/1/14
df['last_contact_correct'] = pd.to_datetime(df['last_contact_correct'])

In [ ]:
df[df['last_contact_correct']<pd.to_datetime(df['cohort_entry_date'])].shape

In [189]:
df = df[df['last_contact_correct']>=pd.to_datetime(df['cohort_entry_date'])]

In [191]:
#remove people who registered with their GP within 1 year of cohort entry 
df['first_contact_date'] = pd.to_datetime(df['first_contact_date'])
df['cohort_entry_date'] = pd.to_datetime(df['cohort_entry_date'])

In [192]:
reg_within_1yr_df = df[df['first_contact_date'] > (df['cohort_entry_date'] - timedelta(days=365))]

In [ ]:
len(list(reg_within_1yr_df['person_id']))

In [194]:
df = df[~df['person_id'].isin(list(reg_within_1yr_df['person_id']))]

In [196]:
#date of death before cohort entry date

In [197]:
df['corrected_death_date'] = pd.to_datetime(df['corrected_death_date'], errors='coerce')

In [198]:
death_date_error = list(df[pd.to_datetime(df['corrected_death_date'])<=df['cohort_entry_date']]['person_id'])

In [ ]:
len(death_date_error)

In [200]:
df = df[~df['person_id'].isin(death_date_error)]

In [201]:
#spurious death dates 
death_date_future = list(df[df['corrected_death_date'].dt.year >= 2025]['person_id'])

In [ ]:
len(death_date_future)

In [203]:
df = df[~df['person_id'].isin(death_date_future)]

In [ ]:
#age at initiation
def age_at_cohort_entry_function(row): 
    print('working on', row.name)
    temp_person = df[df['person_id']==row['person_id']]
    entry_year = float(row['cohort_entry_date'].year)
    dob_year = float(df[df['person_id']==row['person_id']]['year_of_birth'])
    if dob_year == 0: 
        return(np.nan)
    else: 
        age = float(entry_year - dob_year)
        return(age)
    
df['age_at_initiation'] = df.apply(age_at_cohort_entry_function, axis=1)

In [206]:
under_age = list(df[df['age_at_initiation']<18]['person_id'])

In [ ]:
len(under_age)

In [208]:
df = df[~df['person_id'].isin(under_age)]

In [210]:
#covariates
t2dm_codes=list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/t2dm.csv")['code'])
cvd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/cvd_pad.csv")['code'])
copd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH43_copd.csv")['code'])
cf_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH40_cf.csv")['code'])
ms_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH63_ms.csv")['code'])
pd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH77_parkinsons.csv")['code'])
ra_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH80_ra.csv")['code'])
asthma_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH109_asthma.csv")['code'])
epilepsy_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH467_epilepsy.csv")['code'])
dementia_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH473_dementia.csv")['code'])
ibd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH596_ibd.csv")['code'])
hiv_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH1081_hiv_aids.csv")['code'])
liver_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH1085_liver_disease.csv")['code'])
cvd_acute_codes = list(pd.read_csv('phenotypes/phenotype_acute_cvd_event.csv')['code'].unique())

In [211]:
first_obs = first_obs.rename(columns={"observation_code":"code"})

In [ ]:
#t2dm 
t2dm_obs = first_obs[first_obs['code'].isin(t2dm_codes)].dropna(axis=1, how='all')
t2dm_pmh = first_pmh[first_pmh['code'].isin(t2dm_codes)].dropna(axis=1, how='all')
t2dm_contact = first_contact[first_contact['code'].isin(t2dm_codes)].dropna(axis=1, how='all')

t2dm_obs['date'] = pd.to_datetime(t2dm_obs['date'])
t2dm_pmh['date'] = pd.to_datetime(t2dm_pmh['date'])
t2dm_contact['date'] = pd.to_datetime(t2dm_contact['date'])

t2dm_obs_df = t2dm_obs.groupby(['person_id'])['date'].min().reset_index()
t2dm_pmh_df = t2dm_pmh.groupby(['person_id'])['date'].min().reset_index()
t2dm_contact_df = t2dm_contact.groupby(['person_id'])['date'].min().reset_index()

def t2dm_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(t2dm_obs_df['person_id']): 
        person_obs = t2dm_obs_df[t2dm_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(t2dm_pmh_df['person_id']): 
        person_pmh = t2dm_pmh_df[t2dm_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(t2dm_contact_df['person_id']): 
        person_contact = t2dm_contact_df[t2dm_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['t2dm_diagnosis_before_cohort'] = df.apply(t2dm_diagnosis_before_function, axis=1)
df['t2dm_diagnosis_before_cohort'].value_counts(dropna=False)

In [213]:
df = df[df['t2dm_diagnosis_before_cohort']==1]

In [214]:
#ketoacidosis
ketoacidosis_codes = ['INUK.C101.00', 'INUK.C10FN00', 'INUK.C101100', 'INUK.C101y00', 'INUK.C101z00', 'INUK.C103.00','INUK.C103000',
'INUK.C103100','INUK.C103y00','INUK.C103z00', 'INUK.C10FN11', 'INUK.C10FP00', 'INUK.C10FP11']

In [ ]:
#ketoacidosis 
ketoacidosis_obs = first_obs[first_obs['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
#ketoacidosis_pmh = first_pmh[first_pmh['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
ketoacidosis_contact = first_contact[first_contact['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')

ketoacidosis_obs['date'] = pd.to_datetime(ketoacidosis_obs['date'])
#ketoacidosis_pmh['date'] = pd.to_datetime(ketoacidosis_pmh['date'])
ketoacidosis_contact['date'] = pd.to_datetime(ketoacidosis_contact['date'])

ketoacidosis_obs_df = ketoacidosis_obs.groupby(['person_id'])['date'].min().reset_index()
#ketoacidosis_pmh_df = ketoacidosis_pmh.groupby(['person_id'])['date'].min().reset_index()
ketoacidosis_contact_df = ketoacidosis_contact.groupby(['person_id'])['date'].min().reset_index()

def ketoacidosis_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(ketoacidosis_obs_df['person_id']): 
        person_obs = ketoacidosis_obs_df[ketoacidosis_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
   # if person_id in list(ketoacidosis_pmh_df['person_id']): 
      #  person_pmh = ketoacidosis_pmh_df[ketoacidosis_pmh_df['person_id']==person_id]
       # person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
       # if any(person_pmh['date'] <= cohort_entry_date):
          #  res+=1
        #else: 
           # pass 
        
    #contact 
    if person_id in list(ketoacidosis_contact_df['person_id']): 
        person_contact = ketoacidosis_contact_df[ketoacidosis_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['keto_before_cohort'] = df.apply(ketoacidosis_diagnosis_before_function, axis=1)
df['keto_before_cohort'].value_counts(dropna=False)

In [216]:
df = df[df['keto_before_cohort']!= 1]

In [ ]:
#pancreatitis 
pancreatitis_codes = list(pd.read_csv('phenotypes/phenotype_PH232_pancreatitis.csv')['code'].unique())

pancreatitis_obs = first_obs[first_obs['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')
pancreatitis_pmh = first_pmh[first_pmh['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')
pancreatitis_contact = first_contact[first_contact['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')

pancreatitis_obs['date'] = pd.to_datetime(pancreatitis_obs['date'])
pancreatitis_pmh['date'] = pd.to_datetime(pancreatitis_pmh['date'])
pancreatitis_contact['date'] = pd.to_datetime(pancreatitis_contact['date'])

pancreatitis_obs_df = pancreatitis_obs.groupby(['person_id'])['date'].min().reset_index()
pancreatitis_pmh_df = pancreatitis_pmh.groupby(['person_id'])['date'].min().reset_index()
pancreatitis_contact_df = pancreatitis_contact.groupby(['person_id'])['date'].min().reset_index()

def pancreatitis_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(pancreatitis_obs_df['person_id']): 
        person_obs = pancreatitis_obs_df[pancreatitis_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(pancreatitis_pmh_df['person_id']): 
        person_pmh = pancreatitis_pmh_df[pancreatitis_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(pancreatitis_contact_df['person_id']): 
        person_contact = pancreatitis_contact_df[pancreatitis_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['pancreatitis_before_cohort'] = df.apply(pancreatitis_diagnosis_before_function, axis=1)
df['pancreatitis_before_cohort'].value_counts(dropna=False)

In [218]:
df = df[df['pancreatitis_before_cohort']!=1]

In [ ]:
#cvd 
cvd_obs = first_obs[first_obs['code'].isin(cvd_codes)].dropna(axis=1, how='all')
cvd_pmh = first_pmh[first_pmh['code'].isin(cvd_codes)].dropna(axis=1, how='all')
cvd_contact = first_contact[first_contact['code'].isin(cvd_codes)].dropna(axis=1, how='all')

cvd_obs['date'] = pd.to_datetime(cvd_obs['date'])
cvd_pmh['date'] = pd.to_datetime(cvd_pmh['date'])
cvd_contact['date'] = pd.to_datetime(cvd_contact['date'])

cvd_obs_df = cvd_obs.groupby(['person_id'])['date'].min().reset_index()
cvd_pmh_df = cvd_pmh.groupby(['person_id'])['date'].min().reset_index()
cvd_contact_df = cvd_contact.groupby(['person_id'])['date'].min().reset_index()

def cvd_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(cvd_obs_df['person_id']): 
        person_obs = cvd_obs_df[cvd_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(cvd_pmh_df['person_id']): 
        person_pmh = cvd_pmh_df[cvd_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(cvd_contact_df['person_id']): 
        person_contact = cvd_contact_df[cvd_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['cvd_diagnosis_before_cohort'] = df.apply(cvd_diagnosis_before_function, axis=1)
df['cvd_diagnosis_before_cohort'].value_counts(dropna=False)

In [ ]:
#cvd_acute 
cvd_acute_obs = first_obs[first_obs['code'].isin(cvd_acute_codes)].dropna(axis=1, how='all')
cvd_acute_pmh = first_pmh[first_pmh['code'].isin(cvd_acute_codes)].dropna(axis=1, how='all')
cvd_acute_contact = first_contact[first_contact['code'].isin(cvd_acute_codes)].dropna(axis=1, how='all')

cvd_acute_obs['date'] = pd.to_datetime(cvd_acute_obs['date'])
cvd_acute_pmh['date'] = pd.to_datetime(cvd_acute_pmh['date'])
cvd_acute_contact['date'] = pd.to_datetime(cvd_acute_contact['date'])

cvd_acute_obs_df = cvd_acute_obs.groupby(['person_id'])['date'].apply(list).reset_index()
cvd_acute_pmh_df = cvd_acute_pmh.groupby(['person_id'])['date'].apply(list).reset_index()
cvd_acute_contact_df = cvd_acute_contact.groupby(['person_id'])['date'].apply(list).reset_index()

def cvd_acute_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 60)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #obs 
    if person_id in list(cvd_acute_obs_df['person_id']):  
        for date in cvd_acute_obs_df[cvd_acute_obs_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
        
    #pmh 
    if person_id in list(cvd_acute_pmh_df['person_id']): 
        for date in cvd_acute_pmh_df[cvd_acute_pmh_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass 
        
    #contact 
    if person_id in list(cvd_acute_contact_df['person_id']): 
        for date in cvd_acute_contact_df[cvd_acute_contact_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass  
       
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['cvd_acute_diagnosis_before_cohort'] = df.apply(cvd_acute_diagnosis_before_function, axis=1)
df['cvd_acute_diagnosis_before_cohort'].value_counts(dropna=False)

In [223]:
bmi = measure[measure['person_measure_code']=='BMI']

In [224]:
#use raw values 
def bmi_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 10: 
        return(np.nan)
    elif row['raw_value'] > 80: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
bmi['start_change_date'] = pd.to_datetime(bmi['start_change_date'])

In [ ]:
bmi['correct_value'] = bmi.apply(bmi_correct_function, axis=1)

In [227]:
def bmi_at_initiation_function(row): 
    """returns most recent bmi measurement within a window of 540 days"""
    
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 540)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_bmi = bmi[bmi['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_bmi_filter = temp_bmi[(temp_bmi['start_change_date']>= lower_window_date)&(temp_bmi['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_bmi_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['bmi_at_initiation'] = df.apply(bmi_at_initiation_function, axis=1)

In [229]:
def bmi_exclusion_function(row): 
    """Note: if BMI was missing at this point, person was NOT excluded"""
    
    if row['bmi_at_initiation'] > 45: 
        return(1)
    elif pd.isna(row['bmi_at_initiation']): 
        return(0)
    else: 
        return(0)

In [230]:
df['bmi_exclusion'] = df.apply(bmi_exclusion_function, axis=1)

In [ ]:
df['bmi_exclusion'].value_counts(dropna=False)

In [232]:
#HbA1c

In [233]:
bio_change.columns = ['person_id', 'code', 'start_change_date',
       'end_change_date', 'raw_value', 'raw_unit', 'value', 'contact_id',
       'country_code', 'platform_code', 'data_source_code', 'job_id',
       'created_date', 'updated_date', 'status_code']

In [234]:
bio_change = pd.merge(bio_change, thin, how='left', on='code')

In [235]:
hba1c = bio_change[bio_change['label']=='Hb A1C - Diabetic control']

In [ ]:
hba1c['start_change_date'] = pd.to_datetime(hba1c['start_change_date'])

In [237]:
#use raw values 
def hba1c_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 15: 
        return(np.nan)
    elif row['raw_value'] > 515: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
hba1c['hba1c_correct'] = hba1c.apply(hba1c_correct_function, axis=1)

In [239]:
def hba1c_at_initiation_function(row): 
    """Most recent hba1c recorded, within the last 180 days"""
    
    print('working on person', row.name)
    
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 180)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    
    temp_hba1c = hba1c[hba1c['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_hba1c_filter = temp_hba1c[(temp_hba1c['start_change_date']>= lower_window_date)&(temp_hba1c['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_hba1c_filter['hba1c_correct'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])
        print(res_list[0])


In [ ]:
df['hba1c_at_initiation'] = df.apply(hba1c_at_initiation_function, axis=1)

In [241]:
def hba1c_exclusion_function(row): 
    """If HbA1c at initiation is missing, do not remove the person """
    if pd.isna(row['hba1c_at_initiation']): 
        return(0)
    elif row['hba1c_at_initiation'] < 53:
        return(1)
    elif row['hba1c_at_initiation'] > 86: 
        return(1)
    else: 
        return(0)

In [242]:
df['hba1c_excluded'] = df.apply(hba1c_exclusion_function, axis=1)

In [ ]:
df['hba1c_excluded'].value_counts(dropna=False)

In [244]:
#gender

In [245]:
def male_function(row): 
    if str(row['gender_code']).strip() == 'M': 
        return(1)
    else: 
        return(0)

In [246]:
df['male'] = df.apply(male_function, axis=1)

In [247]:
#renal function 

In [248]:
#Recent eGFR < 30 
cr = bio_change[bio_change['code']=='INUK.0024']

In [249]:
#use raw values 
def cr_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 10: 
        return(np.nan)
    elif row['raw_value'] > 1500: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
cr['cr_correct'] = cr.apply(cr_correct_function, axis=1)

In [ ]:
cr['start_change_date'] =  pd.to_datetime(cr['start_change_date'])

In [252]:
def cr_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 540)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_cr = cr[cr['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_cr_filter = temp_cr[(temp_cr['start_change_date']>= lower_window_date)&(temp_cr['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_cr_filter['cr_correct'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['cr_at_initiation'] = df.apply(cr_at_initiation_function, axis=1)

In [254]:
def egfr_at_initiation_function(row): 
    
    if pd.isna(row['cr_at_initiation']):
        return(np.nan)
    else: 
        
        cr = float(row['cr_at_initiation']) # note: units are umol/L
        age = float(row['age_at_initiation'])
        
    #convert units 
        cr_mgdl = cr*0.0113
    
        if row['male'] == 1: 
            egfr = 142*(min(cr_mgdl/0.9, 1)**-0.302)*(max(cr_mgdl/0.9, 1)**-1.2)*(0.9938**age)
            return(egfr)
        
        else:
            egfr = 142*(min(cr_mgdl/0.7, 1)**-0.241)*(max(cr_mgdl/0.7, 1)**-1.2)*(0.9938**age)*1.012
            return(egfr)

In [255]:
df['egfr_at_initiation'] = df.apply(egfr_at_initiation_function, axis=1)

In [256]:
def cr_at_2m_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 60)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_cr = cr[cr['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_cr_filter = temp_cr[(temp_cr['start_change_date']>= lower_window_date)&(temp_cr['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_cr_filter['cr_correct'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['cr_at_2m'] = df.apply(cr_at_2m_function, axis=1)

In [258]:
def egfr_at_2m_function(row): 
    
    if pd.isna(row['cr_at_2m']):
        return(np.nan)
    else: 
        
        cr = float(row['cr_at_2m']) # note: units are umol/L
        age = float(row['age_at_initiation'])
        
    #convert units 
        cr_mgdl = cr*0.0113
    
        if row['male'] == 1: 
            egfr = 142*(min(cr_mgdl/0.9, 1)**-0.302)*(max(cr_mgdl/0.9, 1)**-1.2)*(0.9938**age)
            return(egfr)
        
        else:
            egfr = 142*(min(cr_mgdl/0.7, 1)**-0.241)*(max(cr_mgdl/0.7, 1)**-1.2)*(0.9938**age)*1.012
            return(egfr)

In [259]:
df['egfr_at_2m'] = df.apply(egfr_at_2m_function, axis=1)

In [260]:
def exclude_renal_test_function(row): 
    """Note: if missing - person is still included """
    if pd.isna(row['egfr_at_2m']): 
        return(0)
    elif row['egfr_at_2m'] < 30: 
        return(1)    
    else: 
        return(0)

In [261]:
df['exclude_renal_test'] = df.apply(exclude_renal_test_function, axis=1)

In [ ]:
df['exclude_renal_test'].value_counts(dropna=False)

In [263]:
#liver function tests

In [264]:
alk = bio_change[bio_change['code']=='INUK.001s']
ast = bio_change[bio_change['code']=='INUK.001v']
alt = bio_change[bio_change['code']=='INUK.001u']

In [265]:
#use raw values 
def ast_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 3: 
        return(np.nan)
    elif row['raw_value'] > 950: 
        return(np.nan)
    else: 
        return(row['raw_value'])
    
#use raw values 
def alt_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 3: 
        return(np.nan)
    elif row['raw_value'] > 500: 
        return(np.nan)
    else: 
        return(row['raw_value'])
    
#use raw values 
def alk_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 8: 
        return(np.nan)
    elif row['raw_value'] > 1500: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
ast['correct_value'] = ast.apply(ast_correct_function, axis=1)
alt['correct_value'] = alt.apply(alt_correct_function, axis=1)
alk['correct_value'] = alk.apply(alk_correct_function, axis=1)

In [ ]:
alk['start_change_date'] = pd.to_datetime(alk['start_change_date'])
ast['start_change_date'] = pd.to_datetime(ast['start_change_date'])
alt['start_change_date'] = pd.to_datetime(alt['start_change_date'])

In [268]:
def alt_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 60)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_alt = alt[alt['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_alt_filter = temp_alt[(temp_alt['start_change_date']>= lower_window_date)&(temp_alt['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_alt_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])
    
def alk_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 60)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_alk = alk[alk['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_alk_filter = temp_alk[(temp_alk['start_change_date']>= lower_window_date)&(temp_alk['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_alk_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])
    
def ast_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 60)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_ast = ast[ast['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_ast_filter = temp_ast[(temp_ast['start_change_date']>= lower_window_date)&(temp_ast['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_ast_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['alk_at_initiation'] = df.apply(alk_at_initiation_function, axis=1)

In [ ]:
df['ast_at_initiation'] = df.apply(ast_at_initiation_function, axis=1)

In [ ]:
df['alt_at_initiation'] = df.apply(alt_at_initiation_function, axis=1)

In [272]:
def exclude_liver_test_function(row): 
    if pd.isna(row['ast_at_initiation']) & pd.isna(row['alt_at_initiation']) & pd.isna(row['alk_at_initiation']): 
        return(np.nan)
    elif row['alt_at_initiation'] >= 3*50: 
        return(1)
    elif row['alk_at_initiation'] >= 3*115: 
        return(1)
    elif row['ast_at_initiation'] >= 3*35: 
        return(1)
    else: 
        return(0)

In [273]:
df['exclude_liver_test'] = df.apply(exclude_liver_test_function, axis=1)

In [ ]:
df['exclude_liver_test'].value_counts(dropna=False)

In [275]:
#anti-obesity drugs
anti_obesity_codes = [10814, 120446, 10815]

In [276]:
anti_obesity_dates = complete_pres[complete_pres['product_id'].isin(anti_obesity_codes)].groupby('person_id')['start_date'].apply(list).reset_index()

In [ ]:
#anti-obesity prescribed within the last 90 days before cohort entry  

def anti_obesity_3m_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 90)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #
    if person_id in list(anti_obesity_dates['person_id']):  
        for date in anti_obesity_dates[anti_obesity_dates['person_id']==person_id]['start_date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
    
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['anti_obesity_3m_before'] = df.apply(anti_obesity_3m_before_function, axis=1)
df['anti_obesity_3m_before'].value_counts(dropna=False)

In [278]:
#Recent steroids 
steroid_codes = list(pd.read_csv('drug_names/steroids_product_id.csv')['product_id'])

In [279]:
steroid_dates = complete_pres[complete_pres['product_id'].isin(steroid_codes)].groupby('person_id')['start_date'].apply(list).reset_index()

In [ ]:
#systemic steroid prescribed within 6 weeks of cohort entry  

def steroid_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 6*7)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0


    if person_id in list(steroid_dates['person_id']):  
        for date in steroid_dates[steroid_dates['person_id']==person_id]['start_date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
    
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['steroid_before'] = df.apply(steroid_before_function, axis=1)
df['steroid_before'].value_counts(dropna=False)

In [281]:
#thyroid dose change in 6 weeks prior to cohort entry 
thyroid_drugs = list(pd.read_csv('drug_names/thyroid_product_id.csv')['product_id'])

In [282]:
thyroid_df=complete_pres[(complete_pres['person_id'].isin(df['person_id']))&(complete_pres['product_id'].isin(thyroid_drugs))]

In [ ]:
is_multi = thyroid_df['person_id'].value_counts() > 1
thyroid_df['multiple'] = is_multi

In [ ]:
people_with_dose_change = []


for person in list(thyroid_df['person_id'].unique()): 
    print('person', person)
    
    temp_df = thyroid_df[thyroid_df['person_id']==person]
    
    test_df = temp_df.drop_duplicates(subset=['short_name', 'dosage_1'])
    
    if test_df.shape[0]==1: 
        pass 
    else: 
        people_with_dose_change.append(person)

In [ ]:
thyroid_dose = {}

for person in people_with_dose_change: 
    print(person)
    thyroid_dose[person] = {}
        
    #temp df 
    temp_df = thyroid_df[thyroid_df['person_id']==person][['start_date', 'short_name', 'dosage_1']]
    dates = list(temp_df['start_date'].unique())
    
    for date in dates: 
        dose = temp_df[temp_df['start_date']==date]['dosage_1'].sum()
        thyroid_dose[person][date]=dose
 

In [ ]:
thyroid_date_changes = {}

for person in people_with_dose_change: 
    print(person)
    temp_df = pd.DataFrame.from_dict(thyroid_dose[person], orient='index').reset_index()
    temp_df.columns = ['date', 'dose']
    temp_df['date'] = pd.to_datetime(temp_df['date'])
    temp_df = temp_df.sort_values(by='date', ascending=True)
    temp_df['dose_change']=temp_df['dose'].diff().fillna(0)!=0
    change_dates = list(temp_df[temp_df['dose_change']==True]['date'])
    thyroid_date_changes[person] = change_dates

In [287]:
def thyroid_exclusion_function(row): 
    print('working on', row.name)
    res = 0
    if row['person_id'] not in people_with_dose_change:
        return(0)
    else: 
        
        cohort_entry_date = row['cohort_entry_date']
        lower_window = cohort_entry_date - timedelta(days = 6*7)
        
        for date in thyroid_date_changes: 
            date = pd.to_datetime(date)
            if ((date >= lower_window) & (date <= cohort_entry_date)): 
                res+=1
            else: 
                pass 
        
        if res > 0: 
            return(1)
        else: 
            return(0) 

In [ ]:
df['thyroid_exclusion'] = df.apply(thyroid_exclusion_function, axis=1)

In [ ]:
df['thyroid_exclusion'].value_counts(dropna=False)

In [290]:
#pregnancy within 12 months
preg_codes = list(pd.read_csv('phenotypes/phenotype_PH348_pregnancy.csv')['code'])

In [ ]:
preg_obs = first_obs[first_obs['code'].isin(preg_codes)].dropna(axis=1, how='all')
preg_pmh = first_pmh[first_pmh['code'].isin(preg_codes)].dropna(axis=1, how='all')
preg_contact = first_contact[first_contact['code'].isin(preg_codes)].dropna(axis=1, how='all')

preg_obs['date'] = pd.to_datetime(preg_obs['date'])
preg_pmh['date'] = pd.to_datetime(preg_pmh['date'])
preg_contact['date'] = pd.to_datetime(preg_contact['date'])

preg_obs_df = preg_obs.groupby(['person_id'])['date'].apply(list).reset_index()
preg_pmh_df = preg_pmh.groupby(['person_id'])['date'].apply(list).reset_index()
preg_contact_df = preg_contact.groupby(['person_id'])['date'].apply(list).reset_index()

def preg_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 365)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #obs 
    if person_id in list(preg_obs_df['person_id']):  
        for date in preg_obs_df[preg_obs_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
        
    #pmh 
    if person_id in list(preg_pmh_df['person_id']): 
        for date in preg_pmh_df[preg_pmh_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass 
        
    #contact 
    if person_id in list(preg_contact_df['person_id']): 
        for date in preg_contact_df[preg_contact_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass  
       
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['preg_diagnosis_before_cohort'] = df.apply(preg_diagnosis_before_function, axis=1)
df['preg_diagnosis_before_cohort'].value_counts(dropna=False)

In [292]:
#cancer
cancer_codes = list(pd.read_csv('phenotypes/phenotype_PH616_cancer.csv')['code'])

In [ ]:
#cancer
cancer_obs = first_obs[first_obs['code'].isin(cancer_codes)].dropna(axis=1, how='all')
#cancer_pmh = first_pmh[first_pmh['code'].isin(cancer_codes)].dropna(axis=1, how='all') ##empty
cancer_contact = first_contact[first_contact['code'].isin(cancer_codes)].dropna(axis=1, how='all')

cancer_obs['date'] = pd.to_datetime(cancer_obs['date'])
#cancer_pmh['date'] = pd.to_datetime(cancer_pmh['date'])
cancer_contact['date'] = pd.to_datetime(cancer_contact['date'])

cancer_obs_df = cancer_obs.groupby(['person_id'])['date'].apply(list).reset_index()
#cancer_pmh_df = cancer_pmh.groupby(['person_id'])['date'].apply(list).reset_index()
cancer_contact_df = cancer_contact.groupby(['person_id'])['date'].apply(list).reset_index()

def cancer_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 365.25*5)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #obs 
    if person_id in list(cancer_obs_df['person_id']):  
        for date in cancer_obs_df[cancer_obs_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
        
    #pmh 
    #if person_id in list(cancer_pmh_df['person_id']): 
        #for date in cancer_pmh_df[cancer_pmh_df['person_id']==person_id]['date'].values[0]:
            #if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
               # res+=1
          #  else: 
                #pass 
        
    #contact 
    if person_id in list(cancer_contact_df['person_id']): 
        for date in cancer_contact_df[cancer_contact_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass  
       
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['cancer_diagnosis_5y_before_cohort'] = df.apply(cancer_diagnosis_before_function, axis=1)
df['cancer_diagnosis_5y_before_cohort'].value_counts(dropna=False)

In [294]:
#malaria diagnosis in last 5 years
malaria_codes = ['INUK.1413.00']

In [295]:
malaria_obs = first_obs[first_obs['code'].isin(malaria_codes)].dropna(axis=1, how='all')
malaria_pmh = first_pmh[first_pmh['code'].isin(malaria_codes)].dropna(axis=1, how='all')
malaria_contact = first_contact[first_contact['code'].isin(malaria_codes)].dropna(axis=1, how='all')

In [ ]:
#malaria 
malaria_obs['date'] = pd.to_datetime(malaria_obs['date'])
malaria_pmh['date'] = pd.to_datetime(malaria_pmh['date'])
#malaria_contact['date'] = pd.to_datetime(malaria_contact['date'])

malaria_obs_df = malaria_obs.groupby(['person_id'])['date'].apply(list).reset_index()
malaria_pmh_df = malaria_pmh.groupby(['person_id'])['date'].apply(list).reset_index()
#malaria_contact_df = malaria_contact.groupby(['person_id'])['date'].apply(list).reset_index()

def malaria_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 365.25*5)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #obs 
    if person_id in list(malaria_obs_df['person_id']):  
        for date in malaria_obs_df[malaria_obs_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
        
    #pmh 
    if person_id in list(malaria_pmh_df['person_id']): 
        for date in malaria_pmh_df[malaria_pmh_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass 
        
    #contact 
    #if person_id in list(malaria_contact_df['person_id']): 
       # for date in malaria_contact_df[malaria_contact_df['person_id']==person_id]['date'].values[0]:
            #if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
               # res+=1
           # else: 
                #pass  
       
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['malaria_diagnosis_past_5y'] = df.apply(malaria_diagnosis_before_function, axis=1)
df['malaria_diagnosis_past_5y'].value_counts(dropna=False)

In [297]:
#bariatric surgery
bariatric_codes=['INUK.14NE.00']
bariatric_obs = first_obs[first_obs['code'].isin(bariatric_codes)].dropna(axis=1, how='all')
bariatric_pmh = first_pmh[first_pmh['code'].isin(bariatric_codes)].dropna(axis=1, how='all')
#bariatric_contact = first_contact[first_contact['code'].isin(bariatric_codes)].dropna(axis=1, how='all')

In [ ]:
#bariatric 
bariatric_obs = first_obs[first_obs['code'].isin(bariatric_codes)].dropna(axis=1, how='all')
bariatric_pmh = first_pmh[first_pmh['code'].isin(bariatric_codes)].dropna(axis=1, how='all')
#bariatric_contact = first_contact[first_contact['code'].isin(bariatric_codes)].dropna(axis=1, how='all')

bariatric_obs['date'] = pd.to_datetime(bariatric_obs['date'])
bariatric_pmh['date'] = pd.to_datetime(bariatric_pmh['date'])
#bariatric_contact['date'] = pd.to_datetime(bariatric_contact['date'])

bariatric_obs_df = bariatric_obs.groupby(['person_id'])['date'].min().reset_index()
bariatric_pmh_df = bariatric_pmh.groupby(['person_id'])['date'].min().reset_index()
#bariatric_contact_df = bariatric_contact.groupby(['person_id'])['date'].min().reset_index()

def bariatric_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(bariatric_obs_df['person_id']): 
        person_obs = bariatric_obs_df[bariatric_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(bariatric_pmh_df['person_id']): 
        person_pmh = bariatric_pmh_df[bariatric_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    #if person_id in list(bariatric_contact_df['person_id']): 
      #  person_contact = bariatric_contact_df[bariatric_contact_df['person_id']==person_id]
       ## person_contact['date'] = pd.to_datetime(person_contact['date'])
       # 
       # if any(person_contact['date'] <= cohort_entry_date):
       #     res+=1
       # else: 
          #  pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['bariatric_surgery_before_cohort'] = df.apply(bariatric_diagnosis_before_function, axis=1)
df['bariatric_surgery_before_cohort'].value_counts(dropna=False)

In [299]:
#D&A
drug = list(pd.read_csv('phenotypes/phenotype_PH1108_drug.csv')['code'])
alcohol = list(pd.read_csv('phenotypes/phenotype_PH444_alcohol.csv')['code'])
d_a_codes = drug + alcohol
d_a_obs = first_obs[first_obs['code'].isin(d_a_codes)].dropna(axis=1, how='all')
#d_a_pmh = first_pmh[first_pmh['code'].isin(d_a_codes)].dropna(axis=1, how='all')
d_a_contact = first_contact[first_contact['code'].isin(d_a_codes)].dropna(axis=1, how='all')

In [ ]:
#d_a code only 

#d_a 
d_a_obs = first_obs[first_obs['code'].isin(d_a_codes)].dropna(axis=1, how='all')
#d_a_pmh = first_pmh[first_pmh['code'].isin(d_a_codes)].dropna(axis=1, how='all')
d_a_contact = first_contact[first_contact['code'].isin(d_a_codes)].dropna(axis=1, how='all')

d_a_obs['date'] = pd.to_datetime(d_a_obs['date'])
#d_a_pmh['date'] = pd.to_datetime(d_a_pmh['date'])
d_a_contact['date'] = pd.to_datetime(d_a_contact['date'])

d_a_obs_df = d_a_obs.groupby(['person_id'])['date'].apply(list).reset_index()
#d_a_pmh_df = d_a_pmh.groupby(['person_id'])['date'].apply(list).reset_index()
d_a_contact_df = d_a_contact.groupby(['person_id'])['date'].apply(list).reset_index()

def d_a_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    lower_window_date = pd.to_datetime(row['cohort_entry_date']) - timedelta(days = 90)
    upper_window_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0

    #obs 
    if person_id in list(d_a_obs_df['person_id']):  
        for date in d_a_obs_df[d_a_obs_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date)>=lower_window_date) & (pd.to_datetime(date)<=upper_window_date)):
                res+=1
            else: 
                pass
        
    #pmh 
   # if person_id in list(d_a_pmh_df['person_id']): 
       # for date in d_a_pmh_df[d_a_pmh_df['person_id']==person_id]['date'].values[0]:
         #   if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
             #   res+=1
         #   else: 
                #pass 
        
    #contact 
    if person_id in list(d_a_contact_df['person_id']): 
        for date in d_a_contact_df[d_a_contact_df['person_id']==person_id]['date'].values[0]:
            if ((pd.to_datetime(date) >= lower_window_date) & (pd.to_datetime(date) <= upper_window_date)): 
                res+=1
            else: 
                pass  
       
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['d_a_3m'] = df.apply(d_a_diagnosis_before_function, axis=1)
df['d_a_3m'].value_counts(dropna=False)

In [301]:
#ketoacidosis
ketoacidosis_codes = ['INUK.C101.00', 'INUK.C10FN00', 'INUK.C101100', 'INUK.C101y00', 'INUK.C101z00', 'INUK.C103.00','INUK.C103000',
'INUK.C103100','INUK.C103y00','INUK.C103z00', 'INUK.C10FN11', 'INUK.C10FP00', 'INUK.C10FP11']

In [302]:
ketoacidosis_obs = first_obs[first_obs['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
ketoacidosis_pmh = first_pmh[first_pmh['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
ketoacidosis_contact = first_contact[first_contact['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')

In [ ]:
#ketoacidosis 
ketoacidosis_obs = first_obs[first_obs['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
#ketoacidosis_pmh = first_pmh[first_pmh['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')
ketoacidosis_contact = first_contact[first_contact['code'].isin(ketoacidosis_codes)].dropna(axis=1, how='all')

ketoacidosis_obs['date'] = pd.to_datetime(ketoacidosis_obs['date'])
#ketoacidosis_pmh['date'] = pd.to_datetime(ketoacidosis_pmh['date'])
ketoacidosis_contact['date'] = pd.to_datetime(ketoacidosis_contact['date'])

ketoacidosis_obs_df = ketoacidosis_obs.groupby(['person_id'])['date'].min().reset_index()
#ketoacidosis_pmh_df = ketoacidosis_pmh.groupby(['person_id'])['date'].min().reset_index()
ketoacidosis_contact_df = ketoacidosis_contact.groupby(['person_id'])['date'].min().reset_index()

def ketoacidosis_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(ketoacidosis_obs_df['person_id']): 
        person_obs = ketoacidosis_obs_df[ketoacidosis_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
   # if person_id in list(ketoacidosis_pmh_df['person_id']): 
      #  person_pmh = ketoacidosis_pmh_df[ketoacidosis_pmh_df['person_id']==person_id]
       # person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
       # if any(person_pmh['date'] <= cohort_entry_date):
          #  res+=1
        #else: 
           # pass 
        
    #contact 
    if person_id in list(ketoacidosis_contact_df['person_id']): 
        person_contact = ketoacidosis_contact_df[ketoacidosis_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['keto_before_cohort'] = df.apply(ketoacidosis_diagnosis_before_function, axis=1)
df['keto_before_cohort'].value_counts(dropna=False)

In [304]:
#pancreatitis
pancreatitis_codes = list(pd.read_csv('phenotypes/phenotype_PH232_pancreatitis.csv')['code'].unique())
pancreatitis_obs = first_obs[first_obs['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')
pancreatitis_pmh = first_pmh[first_pmh['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')
pancreatitis_contact = first_contact[first_contact['code'].isin(pancreatitis_codes)].dropna(axis=1, how='all')

In [ ]:
#pancreatitis 
pancreatitis_codes = list(pd.read_csv('phenotypes/phenotype_PH232_pancreatitis.csv')['code'].unique())

pancreatitis_obs['date'] = pd.to_datetime(pancreatitis_obs['date'])
pancreatitis_pmh['date'] = pd.to_datetime(pancreatitis_pmh['date'])
pancreatitis_contact['date'] = pd.to_datetime(pancreatitis_contact['date'])

pancreatitis_obs_df = pancreatitis_obs.groupby(['person_id'])['date'].min().reset_index()
pancreatitis_pmh_df = pancreatitis_pmh.groupby(['person_id'])['date'].min().reset_index()
pancreatitis_contact_df = pancreatitis_contact.groupby(['person_id'])['date'].min().reset_index()

def pancreatitis_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(pancreatitis_obs_df['person_id']): 
        person_obs = pancreatitis_obs_df[pancreatitis_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(pancreatitis_pmh_df['person_id']): 
        person_pmh = pancreatitis_pmh_df[pancreatitis_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(pancreatitis_contact_df['person_id']): 
        person_contact = pancreatitis_contact_df[pancreatitis_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['pancreatitis_before_cohort'] = df.apply(pancreatitis_diagnosis_before_function, axis=1)
df['pancreatitis_before_cohort'].value_counts(dropna=False)

In [306]:
df = df[df['pancreatitis_before_cohort']!=1]

In [307]:
df = df[df['keto_before_cohort']!=1]

In [309]:
#LDL cholesterol
ldl = bio_change[bio_change['code']=='INUK.002F']

In [310]:
def ldl_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 0.2: 
        return(np.nan)
    elif row['raw_value'] > 10: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
ldl['correct_value'] = ldl.apply(ldl_correct_function, axis=1)

In [ ]:
ldl['start_change_date'] = pd.to_datetime(ldl['start_change_date'])

In [313]:
def ldl_at_initiation_function(row): 
    print('working on person', row['person_id'])
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 540)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_ldl = ldl[ldl['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_ldl_filter = temp_ldl[(temp_ldl['start_change_date']>= lower_window_date)&(temp_ldl['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_ldl_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['ldl_at_initiation'] = df.apply(ldl_at_initiation_function, axis=1)

In [315]:
df['ldl_sq'] = df['ldl_at_initiation']*df['ldl_at_initiation']

In [316]:
#HDL 
hdl = bio_change[bio_change['code']=='INUK.002D']

In [317]:
def hdl_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 0.2: 
        return(np.nan)
    elif row['raw_value'] > 5: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
hdl['correct_value'] = hdl.apply(hdl_correct_function, axis=1)

In [ ]:
hdl['start_change_date'] = pd.to_datetime(hdl['start_change_date'])

In [320]:
def hdl_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 540)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_hdl = hdl[hdl['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_hdl_filter = temp_hdl[(temp_hdl['start_change_date']>= lower_window_date)&(temp_hdl['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_hdl_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['hdl_at_initiation'] = df.apply(hdl_at_initiation_function, axis=1)

In [322]:
df['hdl_log'] = np.log(df['hdl_at_initiation'])

In [323]:
#SBP
sbp = measure[measure['person_measure_code']=='TAMAX']

In [324]:
#use raw values 
def sbp_correct_function(row): 
    '''Note biologically plausible values are defined based on distribution in UK Biobank'''
    if row['raw_value'] <= 50: 
        return(np.nan)
    elif row['raw_value'] > 270: 
        return(np.nan)
    else: 
        return(row['raw_value'])

In [ ]:
sbp['correct_value'] = sbp.apply(sbp_correct_function, axis=1)

In [ ]:
sbp['start_change_date'] = pd.to_datetime(sbp['start_change_date'])

In [327]:
def sbp_at_initiation_function(row): 
    print('working on person', row.name)
    
    lower_window_date = row['cohort_entry_date'] - timedelta(days = 540)
    upper_window_date = row['cohort_entry_date'] 
    
    temp_sbp = sbp[sbp['person_id']==row['person_id']].sort_values('start_change_date', ascending=False)
    temp_sbp_filter = temp_sbp[(temp_sbp['start_change_date']>= lower_window_date)&(temp_sbp['start_change_date']<=upper_window_date)]
    
    res_list = list(temp_sbp_filter['correct_value'])
    
    if len(res_list) == 0: 
        return(np.nan)
    else:
        return(res_list[0])

In [ ]:
df['sbp_at_initiation'] = df.apply(sbp_at_initiation_function, axis=1)

In [329]:
#Smoking

In [330]:
smoking = smoking[smoking['person_id'].isin(list(df['person_id']))]

In [331]:
smoking['start_date'] = pd.to_datetime(smoking['start_date'])

In [332]:
def smoking_status_function(row):
    """
    0 - non-smoker
    1 - ex-smoker 
    3 - current smoker 
    
    Only considers recordings of smoking status made before or on the date of cohort entry 
    
    """
    print('working on', row.name)
    
    if row['person_id'] in list(smoking['person_id']): 
        try: 
            cohort_entry_date = df[df['person_id']==row['person_id']][['cohort_entry_date']].iloc[0][0]
        
        
            temp_df = smoking[smoking['person_id']==row['person_id']].sort_values(by='start_date', ascending=False)
            temp_df = temp_df[temp_df['start_date']<=cohort_entry_date]
    
            if temp_df.shape[0] == 1: 
                return(int(temp_df['value']))
    
            elif int(temp_df.iloc[0]['value']) == 3: 
                return(3)
        
            elif 3 in list(temp_df['value']):
                return(1)
        
            elif 1 in list(temp_df['value']): 
                return(1)
        
            else: 
                return(0)
            
        except: 
            return(np.nan)
    
    else: 
        return(np.nan)

In [ ]:
df['smoking_status'] = df.apply(smoking_status_function, axis=1)

In [334]:
#ethnicity

In [335]:
#codes
white = list(eth_count[eth_count['recoded']=='W']['code'])
black = list(eth_count[eth_count['recoded']=='B']['code'])
asian = list(eth_count[eth_count['recoded']=='A']['code'])
mixed = list(eth_count[eth_count['recoded']=='M']['code'])
other = list(eth_count[eth_count['recoded']=='O']['code'])

In [336]:
def eth_code_function(row): 
    print('working on', row.name)
    
    if str(row['ethnicity_code']) in (white): 
        return(0)
    
    elif str(row['ethnicity_code']) in (black): 
        return(1)
    
    elif str(row['ethnicity_code']) in (asian): 
        return(2)
    
    elif str(row['ethnicity_code']) in (mixed): 
        return(3)
    
    elif str(row['ethnicity_code']) in (other): 
        return(4)
    
    else: 
        return(np.nan)

In [ ]:
df['eth_recoded'] = df.apply(eth_code_function, axis=1)

In [338]:
#hf

In [339]:
hf_codes = list(pd.read_csv('phenotypes/phenotype_PH182_hf.csv')['code'].unique())

In [ ]:
#hf 
hf_obs = first_obs[first_obs['code'].isin(hf_codes)].dropna(axis=1, how='all')
hf_pmh = first_pmh[first_pmh['code'].isin(hf_codes)].dropna(axis=1, how='all')
hf_contact = first_contact[first_contact['code'].isin(hf_codes)].dropna(axis=1, how='all')

hf_obs['date'] = pd.to_datetime(hf_obs['date'])
hf_pmh['date'] = pd.to_datetime(hf_pmh['date'])
hf_contact['date'] = pd.to_datetime(hf_contact['date'])

hf_obs_df = hf_obs.groupby(['person_id'])['date'].min().reset_index()
hf_pmh_df = hf_pmh.groupby(['person_id'])['date'].min().reset_index()
hf_contact_df = hf_contact.groupby(['person_id'])['date'].min().reset_index()

def hf_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(hf_obs_df['person_id']): 
        person_obs = hf_obs_df[hf_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(hf_pmh_df['person_id']): 
        person_pmh = hf_pmh_df[hf_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(hf_contact_df['person_id']): 
        person_contact = hf_contact_df[hf_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['hf'] = df.apply(hf_diagnosis_before_function, axis=1)
df['hf'].value_counts(dropna=False)

In [341]:
#smi
schiz = pd.read_csv('phenotypes/phenotype_PH285_schiz_correct.csv')
dep = pd.read_csv('phenotypes/phenotype_PH149_dep_correct.csv')
bipolar = pd.read_csv('phenotypes/phenotype_PH38_bipolar_correct.csv')

schiz_codes = list(schiz['code'])
dep_codes = list(dep['code'])
bipolar_codes = list(bipolar['code'])
smi_codes = schiz_codes + dep_codes + bipolar_codes

In [ ]:
#smi 
smi_obs = first_obs[first_obs['code'].isin(smi_codes)].dropna(axis=1, how='all')
smi_pmh = first_pmh[first_pmh['code'].isin(smi_codes)].dropna(axis=1, how='all')
smi_contact = first_contact[first_contact['code'].isin(smi_codes)].dropna(axis=1, how='all')

smi_obs['date'] = pd.to_datetime(smi_obs['date'])
smi_pmh['date'] = pd.to_datetime(smi_pmh['date'])
smi_contact['date'] = pd.to_datetime(smi_contact['date'])

smi_obs_df = smi_obs.groupby(['person_id'])['date'].min().reset_index()
smi_pmh_df = smi_pmh.groupby(['person_id'])['date'].min().reset_index()
smi_contact_df = smi_contact.groupby(['person_id'])['date'].min().reset_index()

def smi_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(smi_obs_df['person_id']): 
        person_obs = smi_obs_df[smi_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(smi_pmh_df['person_id']): 
        person_pmh = smi_pmh_df[smi_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(smi_contact_df['person_id']): 
        person_contact = smi_contact_df[smi_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['smi'] = df.apply(smi_diagnosis_before_function, axis=1)
df['smi'].value_counts(dropna=False)

In [343]:
#remaining comorbidities 
copd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH43_copd.csv")['code'])
cf_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH40_cf.csv")['code'])
ms_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH63_ms.csv")['code'])
pd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH77_parkinsons.csv")['code'])
ra_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH80_ra.csv")['code'])
asthma_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH109_asthma.csv")['code'])
epilepsy_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH467_epilepsy.csv")['code'])
dementia_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH473_dementia.csv")['code'])
ibd_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH596_ibd.csv")['code'])
hiv_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH1081_hiv_aids.csv")['code'])
liver_codes = list(pd.read_csv("S:/23_010_CVD_trial_emu/phenotypes/phenotype_PH1085_liver_disease.csv")['code'])

In [ ]:
#copd 
copd_obs = first_obs[first_obs['code'].isin(copd_codes)].dropna(axis=1, how='all')
copd_pmh = first_pmh[first_pmh['code'].isin(copd_codes)].dropna(axis=1, how='all')
copd_contact = first_contact[first_contact['code'].isin(copd_codes)].dropna(axis=1, how='all')

copd_obs['date'] = pd.to_datetime(copd_obs['date'])
copd_pmh['date'] = pd.to_datetime(copd_pmh['date'])
copd_contact['date'] = pd.to_datetime(copd_contact['date'])

copd_obs_df = copd_obs.groupby(['person_id'])['date'].min().reset_index()
copd_pmh_df = copd_pmh.groupby(['person_id'])['date'].min().reset_index()
copd_contact_df = copd_contact.groupby(['person_id'])['date'].min().reset_index()

def copd_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(copd_obs_df['person_id']): 
        person_obs = copd_obs_df[copd_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(copd_pmh_df['person_id']): 
        person_pmh = copd_pmh_df[copd_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(copd_contact_df['person_id']): 
        person_contact = copd_contact_df[copd_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['copd'] = df.apply(copd_diagnosis_before_function, axis=1)
df['copd'].value_counts(dropna=False)

In [345]:
#MS
ms_obs = first_obs[first_obs['code'].isin(ms_codes)].dropna(axis=1, how='all')
ms_pmh = first_pmh[first_pmh['code'].isin(ms_codes)].dropna(axis=1, how='all')
ms_contact = first_contact[first_contact['code'].isin(ms_codes)].dropna(axis=1, how='all')

In [ ]:
#ms 

ms_obs_df = ms_obs.groupby(['person_id'])['date'].min().reset_index()
#ms_pmh_df = ms_pmh.groupby(['person_id'])['date'].min().reset_index()
ms_contact_df = ms_contact.groupby(['person_id'])['date'].min().reset_index()

def ms_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(ms_obs_df['person_id']): 
        person_obs = ms_obs_df[ms_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    #if person_id in list(ms_pmh_df['person_id']): 
       # person_pmh = ms_pmh_df[ms_pmh_df['person_id']==person_id]
        #person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
       # if any(person_pmh['date'] <= cohort_entry_date):
       #     res+=1
      #  else: 
        #    pass 
        
    #contact 
    if person_id in list(ms_contact_df['person_id']): 
        person_contact = ms_contact_df[ms_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['ms'] = df.apply(ms_diagnosis_before_function, axis=1)
df['ms'].value_counts(dropna=False)

In [ ]:
#pd 
pd_obs = first_obs[first_obs['code'].isin(pd_codes)].dropna(axis=1, how='all')
pd_pmh = first_pmh[first_pmh['code'].isin(pd_codes)].dropna(axis=1, how='all')
pd_contact = first_contact[first_contact['code'].isin(pd_codes)].dropna(axis=1, how='all')

pd_obs['date'] = pd.to_datetime(pd_obs['date'])
pd_pmh['date'] = pd.to_datetime(pd_pmh['date'])
pd_contact['date'] = pd.to_datetime(pd_contact['date'])

pd_obs_df = pd_obs.groupby(['person_id'])['date'].min().reset_index()
pd_pmh_df = pd_pmh.groupby(['person_id'])['date'].min().reset_index()
pd_contact_df = pd_contact.groupby(['person_id'])['date'].min().reset_index()

def pd_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(pd_obs_df['person_id']): 
        person_obs = pd_obs_df[pd_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(pd_pmh_df['person_id']): 
        person_pmh = pd_pmh_df[pd_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(pd_contact_df['person_id']): 
        person_contact = pd_contact_df[pd_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['pd'] = df.apply(pd_diagnosis_before_function, axis=1)
df['pd'].value_counts(dropna=False)

In [ ]:
#ra 
ra_obs = first_obs[first_obs['code'].isin(ra_codes)].dropna(axis=1, how='all')
ra_pmh = first_pmh[first_pmh['code'].isin(ra_codes)].dropna(axis=1, how='all')
ra_contact = first_contact[first_contact['code'].isin(ra_codes)].dropna(axis=1, how='all')

ra_obs['date'] = pd.to_datetime(ra_obs['date'])
ra_pmh['date'] = pd.to_datetime(ra_pmh['date'])
ra_contact['date'] = pd.to_datetime(ra_contact['date'])

ra_obs_df = ra_obs.groupby(['person_id'])['date'].min().reset_index()
ra_pmh_df = ra_pmh.groupby(['person_id'])['date'].min().reset_index()
ra_contact_df = ra_contact.groupby(['person_id'])['date'].min().reset_index()

def ra_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(ra_obs_df['person_id']): 
        person_obs = ra_obs_df[ra_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(ra_pmh_df['person_id']): 
        person_pmh = ra_pmh_df[ra_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(ra_contact_df['person_id']): 
        person_contact = ra_contact_df[ra_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['ra'] = df.apply(ra_diagnosis_before_function, axis=1)
df['ra'].value_counts(dropna=False)

In [ ]:
#asthma 
asthma_obs = first_obs[first_obs['code'].isin(asthma_codes)].dropna(axis=1, how='all')
asthma_pmh = first_pmh[first_pmh['code'].isin(asthma_codes)].dropna(axis=1, how='all')
asthma_contact = first_contact[first_contact['code'].isin(asthma_codes)].dropna(axis=1, how='all')

asthma_obs['date'] = pd.to_datetime(asthma_obs['date'])
asthma_pmh['date'] = pd.to_datetime(asthma_pmh['date'])
asthma_contact['date'] = pd.to_datetime(asthma_contact['date'])

asthma_obs_df = asthma_obs.groupby(['person_id'])['date'].min().reset_index()
asthma_pmh_df = asthma_pmh.groupby(['person_id'])['date'].min().reset_index()
asthma_contact_df = asthma_contact.groupby(['person_id'])['date'].min().reset_index()

def asthma_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(asthma_obs_df['person_id']): 
        person_obs = asthma_obs_df[asthma_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(asthma_pmh_df['person_id']): 
        person_pmh = asthma_pmh_df[asthma_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(asthma_contact_df['person_id']): 
        person_contact = asthma_contact_df[asthma_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['asthma'] = df.apply(asthma_diagnosis_before_function, axis=1)
df['asthma'].value_counts(dropna=False)

In [350]:
#epilepsy
epilepsy_obs = first_obs[first_obs['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')
epilepsy_pmh = first_pmh[first_pmh['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')
epilepsy_contact = first_contact[first_contact['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')

In [ ]:
#epilepsy 
epilepsy_obs = first_obs[first_obs['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')
#epilepsy_pmh = first_pmh[first_pmh['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')
epilepsy_contact = first_contact[first_contact['code'].isin(epilepsy_codes)].dropna(axis=1, how='all')

epilepsy_obs['date'] = pd.to_datetime(epilepsy_obs['date'])
#epilepsy_pmh['date'] = pd.to_datetime(epilepsy_pmh['date'])
epilepsy_contact['date'] = pd.to_datetime(epilepsy_contact['date'])

epilepsy_obs_df = epilepsy_obs.groupby(['person_id'])['date'].min().reset_index()
#epilepsy_pmh_df = epilepsy_pmh.groupby(['person_id'])['date'].min().reset_index()
epilepsy_contact_df = epilepsy_contact.groupby(['person_id'])['date'].min().reset_index()

def epilepsy_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(epilepsy_obs_df['person_id']): 
        person_obs = epilepsy_obs_df[epilepsy_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    #if person_id in list(epilepsy_pmh_df['person_id']): 
        #person_pmh = epilepsy_pmh_df[epilepsy_pmh_df['person_id']==person_id]
        #person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        #if any(person_pmh['date'] <= cohort_entry_date):
            #res+=1
        #else: 
            #pass 
        
    #contact 
    if person_id in list(epilepsy_contact_df['person_id']): 
        person_contact = epilepsy_contact_df[epilepsy_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['epilepsy'] = df.apply(epilepsy_diagnosis_before_function, axis=1)
df['epilepsy'].value_counts(dropna=False)

In [352]:
#dementia
dementia_obs = first_obs[first_obs['code'].isin(dementia_codes)].dropna(axis=1, how='all')
dementia_pmh = first_pmh[first_pmh['code'].isin(dementia_codes)].dropna(axis=1, how='all')
dementia_contact = first_contact[first_contact['code'].isin(dementia_codes)].dropna(axis=1, how='all')

In [ ]:
#dementia
dementia_obs['date'] = pd.to_datetime(dementia_obs['date'])
#dementia_pmh['date'] = pd.to_datetime(dementia_pmh['date'])
dementia_contact['date'] = pd.to_datetime(dementia_contact['date'])

dementia_obs_df = dementia_obs.groupby(['person_id'])['date'].min().reset_index()
#dementia_pmh_df = dementia_pmh.groupby(['person_id'])['date'].min().reset_index()
dementia_contact_df = dementia_contact.groupby(['person_id'])['date'].min().reset_index()

def dementia_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(dementia_obs_df['person_id']): 
        person_obs = dementia_obs_df[dementia_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    #if person_id in list(dementia_pmh_df['person_id']): 
       # person_pmh = dementia_pmh_df[dementia_pmh_df['person_id']==person_id]
       # person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
       # if any(person_pmh['date'] <= cohort_entry_date):
       #     res+=1
       # else: 
       #     pass 
        
    #contact 
    if person_id in list(dementia_contact_df['person_id']): 
        person_contact = dementia_contact_df[dementia_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['dementia'] = df.apply(dementia_diagnosis_before_function, axis=1)
df['dementia'].value_counts(dropna=False)

In [354]:
#ibd
ibd_obs = first_obs[first_obs['code'].isin(ibd_codes)].dropna(axis=1, how='all')
ibd_pmh = first_pmh[first_pmh['code'].isin(ibd_codes)].dropna(axis=1, how='all')
ibd_contact = first_contact[first_contact['code'].isin(ibd_codes)].dropna(axis=1, how='all')

In [ ]:
#ibd 
ibd_obs['date'] = pd.to_datetime(ibd_obs['date'])
#ibd_pmh['date'] = pd.to_datetime(ibd_pmh['date'])
ibd_contact['date'] = pd.to_datetime(ibd_contact['date'])

ibd_obs_df = ibd_obs.groupby(['person_id'])['date'].min().reset_index()
#ibd_pmh_df = ibd_pmh.groupby(['person_id'])['date'].min().reset_index()
ibd_contact_df = ibd_contact.groupby(['person_id'])['date'].min().reset_index()

def ibd_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(ibd_obs_df['person_id']): 
        person_obs = ibd_obs_df[ibd_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    #if person_id in list(ibd_pmh_df['person_id']): 
       # person_pmh = ibd_pmh_df[ibd_pmh_df['person_id']==person_id]
       # person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
       # if any(person_pmh['date'] <= cohort_entry_date):
       #     res+=1
       # else: 
       #     pass 
        
    #contact 
    if person_id in list(ibd_contact_df['person_id']): 
        person_contact = ibd_contact_df[ibd_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['ibd'] = df.apply(ibd_diagnosis_before_function, axis=1)
df['ibd'].value_counts(dropna=False)

In [ ]:
#liver 
liver_obs = first_obs[first_obs['code'].isin(liver_codes)].dropna(axis=1, how='all')
liver_pmh = first_pmh[first_pmh['code'].isin(liver_codes)].dropna(axis=1, how='all')
liver_contact = first_contact[first_contact['code'].isin(liver_codes)].dropna(axis=1, how='all')

liver_obs['date'] = pd.to_datetime(liver_obs['date'])
liver_pmh['date'] = pd.to_datetime(liver_pmh['date'])
liver_contact['date'] = pd.to_datetime(liver_contact['date'])

liver_obs_df = liver_obs.groupby(['person_id'])['date'].min().reset_index()
liver_pmh_df = liver_pmh.groupby(['person_id'])['date'].min().reset_index()
liver_contact_df = liver_contact.groupby(['person_id'])['date'].min().reset_index()

def liver_diagnosis_before_function(row): 
    print(f'working on person {row.name}')
    
    #extract relevant data 
    person_id = row['person_id']
    cohort_entry_date = pd.to_datetime(row['cohort_entry_date'])
    res = 0 

    
    #obs 
    if person_id in list(liver_obs_df['person_id']): 
        person_obs = liver_obs_df[liver_obs_df['person_id']==person_id]
        person_obs['date'] = pd.to_datetime(person_obs['date'])
        
        if any(person_obs['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #pmh 
    if person_id in list(liver_pmh_df['person_id']): 
        person_pmh = liver_pmh_df[liver_pmh_df['person_id']==person_id]
        person_pmh['date'] = pd.to_datetime(person_pmh['date'])
        
        if any(person_pmh['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass 
        
    #contact 
    if person_id in list(liver_contact_df['person_id']): 
        person_contact = liver_contact_df[liver_contact_df['person_id']==person_id]
        person_contact['date'] = pd.to_datetime(person_contact['date'])
        
        if any(person_contact['date'] <= cohort_entry_date):
            res+=1
        else: 
            pass  
    
        
    #final count 
    if res > 0: 
        return(1)
    else: 
        return(0)
    
df['liver'] = df.apply(liver_diagnosis_before_function, axis=1)
df['liver'].value_counts(dropna=False)

In [357]:
def su_before_initiation_function(row): 
    if row['su_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['su_before_initiation'] = df.apply(su_before_initiation_function, axis=1)

In [358]:
def insulin_before_initiation_function(row): 
    if row['insulin_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['insulin_before_initiation'] = df.apply(insulin_before_initiation_function, axis=1)

In [ ]:
def anti_htn_before_initiation_function(row): 
    if row['anti_htn_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['anti_htn_before_initiation'] = df.apply(anti_htn_before_initiation_function, axis=1)

df['anti_htn_before_initiation'].value_counts(dropna=False)

In [ ]:
def metformin_monotherapy_before_initiation_function(row): 
    if row['metformin_monotherapy_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['metformin_monotherapy_before_initiation'] = df.apply(metformin_monotherapy_before_initiation_function, axis=1)

df['metformin_monotherapy_before_initiation'].value_counts(dropna=False)

In [ ]:
def lipid_lowering_before_initiation_function(row): 
    if row['lipid_lowering_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['lipid_lowering_before_initiation'] = df.apply(lipid_lowering_before_initiation_function, axis=1)

df['lipid_lowering_before_initiation'].value_counts(dropna=False)

In [ ]:
def anti_platelet_before_initiation_function(row): 
    if row['anti_platelet_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['anti_platelet_before_initiation'] = df.apply(anti_platelet_before_initiation_function, axis=1)

df['anti_platelet_before_initiation'].value_counts(dropna=False)

In [ ]:
def anticoag_before_initiation_function(row): 
    if row['anticoag_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['anticoag_before_initiation'] = df.apply(anticoag_before_initiation_function, axis=1)

df['anticoag_before_initiation'].value_counts(dropna=False)

In [ ]:
def glp1ra_before_initiation_function(row): 
    if row['glp1ra_first_dates'] <= row['cohort_entry_date']: 
        return(1)
    else: 
        return(0)
    
df['glp1ra_before_initiation'] = df.apply(glp1ra_before_initiation_function, axis=1)

df['glp1ra_before_initiation'].value_counts(dropna=False)

In [365]:
#eligibility criteria

In [ ]:
df['hba1c_excluded'].value_counts(dropna=False)

In [367]:
def rct_ineligible_function(row): 
    total = 0 
    
    if row['hba1c_excluded'] == 1: 
        total +=1
    else: 
        pass 
    
    if row['bmi_exclusion'] == 1: 
        total+=1    
    else: 
        pass 
    
    if row['cvd_diagnosis_before_cohort'] == 0: 
        total += 1
    else: 
        pass
    
    if row['cvd_acute_diagnosis_before_cohort'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['exclude_renal_test'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['anti_obesity_3m_before'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['steroid_before'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['preg_diagnosis_before_cohort'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['bariatric_surgery_before_cohort'] == 1: 
        total += 1 
    else: 
        pass 
    
    if row['malaria_diagnosis_past_5y'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['thyroid_exclusion'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['d_a_3m'] == 1: 
        total += 1
    else: 
        pass 
    
    if total >0: 
        return(1)
    
    else: 
        return(0)
    
df['rct_ineligible'] = df.apply(rct_ineligible_function, axis = 1)


In [ ]:
def rct_ineligible_number_function(row): 
    print("working on:", row['person_id'])
    total = 0 
    
    if row['hba1c_excluded'] == 1: 
        total +=1
    else: 
        pass 
    
    if row['bmi_exclusion'] == 1: 
        total+=1    
    else: 
        pass 
    
    if row['cvd_diagnosis_before_cohort'] == 0: 
        total += 1
    else: 
        pass
    
    if row['cvd_acute_diagnosis_before_cohort'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['exclude_renal_test'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['anti_obesity_3m_before'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['steroid_before'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['preg_diagnosis_before_cohort'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['bariatric_surgery_before_cohort'] == 1: 
        total += 1 
    else: 
        pass 
    
    if row['malaria_diagnosis_past_5y'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['thyroid_exclusion'] == 1: 
        total += 1
    else: 
        pass 
    
    if row['d_a_3m'] == 1: 
        total += 1
    else: 
        pass 
    
    return(total)
    
df['rct_ineligible_reason_number'] = df.apply(rct_ineligible_number_function, axis = 1)


In [ ]:
df['rct_ineligible_reason_number'].value_counts(dropna=False)

In [368]:
#empa 
df['cohort'].value_counts(dropna=False)

def empa_function(row): 
    if row['cohort'] == 'included - empa': 
        return(1)
    else: 
        return(0)
    
df['empa'] = df.apply(empa_function, axis=1)

In [ ]:
df['empa'].value_counts(dropna=False)

In [370]:
df['cohort_year'] = df['cohort_entry_date'].dt.year

In [371]:
#cohort year recoded 
df['cohort_year_recoded'] = None 

df.loc[df['cohort_year']<= 2016, 'cohort_year_recoded'] = 0
df.loc[(df['cohort_year']> 2016) & (df['cohort_year']<2020), 'cohort_year_recoded'] = 1
df.loc[df['cohort_year']>= 2020, 'cohort_year_recoded'] = 2

df['cohort_year_recoded'] = df['cohort_year_recoded'].astype(int)